## 1. Data Collection & Dataset Overview

In this study, we analyze how poverty levels and ethnic composition affect mortality rates across U.S. counties. The objective is to examine whether socioeconomic and demographic factors are associated with differences in health outcomes.

### Data Sources:

1. **U.S. Census Bureau (Poverty Data):** County-level poverty rates, representing the percentage of the population living below the poverty line.
2. **U.S. Census Bureau (Demographics Data):** County-level ethnic composition, including the population shares of Black, White, and Hispanic groups.
3. **Kaggle (Mortality Data):** County-level mortality rates (deaths per 100,000 individuals), compiled from publicly available health statistics.

### Dataset Characteristics:

The final dataset is constructed by merging three datasets using county-level identifiers. It contains the following key variables:

- **FIPS:** Unique county identifier used to merge datasets.
- **State & County:** Geographic identifiers for each observation.
- **poverty_rate:** Percentage of the population living below the poverty line (Census poverty dataset).
- **black, white, hispanic:** Share of each ethnic group in the county population (%) (Census demographics dataset).
- **mortality_rate:** Number of deaths per 100,000 individuals (Kaggle dataset).

The merged dataset includes approximately 3,000 U.S. counties and provides a comprehensive view of socioeconomic conditions, demographic composition, and mortality outcomes. This structure enables both exploratory analysis and statistical testing of the relationship between poverty, ethnicity, and mortality.

In [ ]:
import pandas as pd

pov = pd.read_csv("datasets/poverty2010.csv", skiprows=2)
pov.columns = pov.columns.str.strip().str.lower()

pov = pov[[
    "state fips",
    "county fips",
    "postal",
    "name",
    "poverty percent all ages"
]].copy()

pov["state fips"] = pd.to_numeric(pov["state fips"], errors="coerce")
pov["county fips"] = pd.to_numeric(pov["county fips"], errors="coerce")
pov["poverty percent all ages"] = pd.to_numeric(pov["poverty percent all ages"], errors="coerce")

pov = pov.dropna(subset=["state fips", "county fips", "poverty percent all ages"])

pov = pov[pov["county fips"] != 0].copy()

pov["state fips"] = pov["state fips"].astype(int).astype(str).str.zfill(2)
pov["county fips"] = pov["county fips"].astype(int).astype(str).str.zfill(3)
pov["fips"] = pov["state fips"] + pov["county fips"]

pov = pov[["fips", "postal", "name", "poverty percent all ages"]].copy()
pov.columns = ["fips", "state", "county", "poverty_rate"]

pov = pov[pov["county"].str.lower().str.strip() != "united states"].copy()

pov["county"] = pov["county"].astype(str).str.strip()
pov["state"] = pov["state"].astype(str).str.strip().str.upper()


pov.head()

In [ ]:
import pandas as pd
import re

demo = pd.read_csv("datasets/demographics.csv")
demo.columns = demo.columns.str.strip().str.lower()

demo = demo[[
    "county",
    "state",
    "ethnicities.black alone",
    "ethnicities.white alone",
    "ethnicities.hispanic or latino"
]].copy()

demo.columns = ["county", "state", "black", "white", "hispanic"]

demo["black"] = pd.to_numeric(demo["black"], errors="coerce")
demo["white"] = pd.to_numeric(demo["white"], errors="coerce")
demo["hispanic"] = pd.to_numeric(demo["hispanic"], errors="coerce")

for col in ["black", "white", "hispanic"]:
    demo.loc[(demo[col] < 0) | (demo[col] > 100), col] = pd.NA

demo = demo.dropna(subset=["black", "white", "hispanic"])

demo["county"] = demo["county"].astype(str).str.strip()
demo["state"] = demo["state"].astype(str).str.strip().str.upper()

def normalize_county(x):
    x = str(x).lower().strip()
    x = re.sub(r' county$', '', x)
    x = re.sub(r' parish$', '', x)
    x = re.sub(r' borough$', '', x)
    x = re.sub(r' census area$', '', x)
    x = re.sub(r' municipality$', '', x)
    x = re.sub(r' city and borough$', '', x)
    x = re.sub(r'[^a-z ]', '', x)
    x = re.sub(r'\s+', ' ', x)
    return x

demo["county_clean"] = demo["county"].apply(normalize_county)


demo.head()

In [ ]:
import pandas as pd


mort = pd.read_csv("datasets/mortality.csv")
mort.columns = mort.columns.str.strip().str.lower()

mort = mort[[
    "fips",
    "category",
    "mortality rate, 2010*"
]].copy()

mort.columns = ["fips", "category", "mortality_rate"]


mort["fips"] = pd.to_numeric(mort["fips"], errors="coerce")
mort["mortality_rate"] = pd.to_numeric(mort["mortality_rate"], errors="coerce")

mort = mort.dropna(subset=["fips", "mortality_rate"])

mort["fips"] = mort["fips"].astype(int).astype(str).str.zfill(5)

mort["category"] = mort["category"].astype(str).str.strip()


mort = mort[mort["category"] == "Cardiovascular diseases"].copy()


mort = mort.set_index("fips")


mort.head()

In [ ]:
import pandas as pd
import re

def normalize_county(x):
    x = str(x).lower().strip()
    x = re.sub(r' county$', '', x)
    x = re.sub(r' parish$', '', x)
    x = re.sub(r' borough$', '', x)
    x = re.sub(r' census area$', '', x)
    x = re.sub(r' municipality$', '', x)
    x = re.sub(r' city and borough$', '', x)
    x = re.sub(r'[^a-z ]', '', x)
    x = re.sub(r'\s+', ' ', x)
    return x

# clean county/state fields
pov["county"] = pov["county"].astype(str).str.strip()
pov["state"] = pov["state"].astype(str).str.strip().str.upper()
pov["county_clean"] = pov["county"].apply(normalize_county)

demo["county"] = demo["county"].astype(str).str.strip()
demo["state"] = demo["state"].astype(str).str.strip().str.upper()
demo["county_clean"] = demo["county"].apply(normalize_county)

# merge poverty + demographics
base = pd.merge(
    pov,
    demo,
    on=["state", "county_clean"],
    how="inner",
    suffixes=("_pov", "_demo")
)

base = base[[
    "fips",
    "state",
    "county_pov",
    "poverty_rate",
    "black",
    "white",
    "hispanic"
]].copy()

base.columns = [
    "fips",
    "state",
    "county",
    "poverty_rate",
    "black",
    "white",
    "hispanic"
]

# add mortality
data = pd.merge(
    base,
    mort,
    on="fips",
    how="inner"
)

# final column order
data = data[[
    "fips",
    "state",
    "county",
    "poverty_rate",
    "black",
    "white",
    "hispanic",
    "mortality_rate"
]].copy()

pd.set_option("display.max_rows", 10)
pd.set_option("display.min_rows", 5)
pd.set_option("display.max_columns", None)

data

## 2. Exploratory Data Analysis (EDA)

In this section, we examine the relationships between poverty, ethnic composition, and mortality across U.S. counties. We first assess data quality, then explore overall patterns using summary statistics, and finally visualize how these variables are associated with each other.

### Step 1 - Checking Missing Values

Before starting the exploratory data analysis, I first checked whether the merged dataset contains any missing values. This step is important because missing observations can distort summary statistics, visualizations, and later regression results.

At this stage, the goal is to understand the general health of the dataset and see whether any variables require cleaning before analysis.

In [ ]:
missing = data.isnull().sum().reset_index()
missing.columns = ["Variable", "Missing Count"]
print(missing)

## Step 2 Summary Statistics and Extreme Values

The summary statistics indicate substantial variation across U.S. counties in both poverty and mortality rates. The average poverty rate is approximately 16.8%, while mortality rates average around 278 deaths per 100,000 individuals, suggesting notable disparities in socioeconomic and health outcomes.

The extreme values further highlight these differences. Ziebach County (SD) exhibits the highest poverty rate at 50.1%, whereas Falls Church city (VA) has the lowest at 3.1%. Similarly, Buffalo County (SD) records the highest mortality rate (539.93), while Pitkin County (CO) shows the lowest (79.29).

These findings suggest that certain counties experience significantly worse socioeconomic and health conditions, motivating further investigation into the relationship between poverty, demographic composition, and mortality.

In [ ]:


import pandas as pd

summary = data.describe().T[["mean", "std", "min", "max"]].round(2)

print("\n=== SUMMARY STATISTICS ===")
print(summary.to_string())


max_pov = data.loc[data["poverty_rate"].idxmax()]
min_pov = data.loc[data["poverty_rate"].idxmin()]
max_mort = data.loc[data["mortality_rate"].idxmax()]
min_mort = data.loc[data["mortality_rate"].idxmin()]

extremes = pd.DataFrame({
    "Metric": [
        "Highest Poverty",
        "Lowest Poverty",
        "Highest Mortality",
        "Lowest Mortality"
    ],
    "State": [
        max_pov["state"],
        min_pov["state"],
        max_mort["state"],
        min_mort["state"]
    ],
    "County": [
        max_pov["county"],
        min_pov["county"],
        max_mort["county"],
        min_mort["county"]
    ],
    "Value": [
        max_pov["poverty_rate"],
        min_pov["poverty_rate"],
        max_mort["mortality_rate"],
        min_mort["mortality_rate"]
    ]
})

print("\n=== EXTREME VALUES ===")
print(extremes.to_string(index=False))

### Step 3 - Poverty vs Mortality

This scatter plot visualizes the relationship between poverty rate and mortality rate across counties. It helps assess whether higher poverty is associated with worse health outcomes.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set(style="whitegrid")

plt.figure(figsize=(10,6))

sns.scatterplot(
    x="poverty_rate",
    y="mortality_rate",
    data=data,
    alpha=0.6
)

plt.title("Poverty vs Mortality Across U.S. Counties")
plt.xlabel("Poverty Rate (%)")
plt.ylabel("Mortality Rate (per 100k people)")

plt.yticks(np.arange(0, 601, 100))  # 0,100,200,...600

# istersen x-axis de sabitle
plt.xticks(np.arange(0, 51, 5))  # 0,5,10,...50

plt.tight_layout()
plt.show()


### Step 4 - Ethnic Composition and Mortality

This figure examines how the share of different ethnic groups (Black, White, and Hispanic) is associated with mortality rates across counties.

Each point represents a county, while the lines show the overall trend between the population share of each group and mortality.

The results suggest that:
- Counties with higher Black population shares tend to exhibit higher mortality rates (positive relationship).
- Counties with higher White and Hispanic population shares show a weaker or slightly negative relationship with mortality.

Overall, while poverty remains a key driver of mortality, the trends indicate that ethnic composition may also be associated with differences in health outcomes across regions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10,6))

ethnicities = ["black", "white", "hispanic"]
colors = ["red", "blue", "green"]

for eth, color in zip(ethnicities, colors):
    x = data[eth]
    y = data["mortality_rate"]
    
    # scatter
    plt.scatter(x, y, alpha=0.2, s=10, color=color)
    
    # trend line
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = m * x_line + b
    
    plt.plot(x_line, y_line, color=color, label=eth.capitalize())

plt.xlabel("Ethnic Population Share (%)")
plt.ylabel("Mortality Rate (per 100k)")
plt.title("Ethnic Composition vs Mortality")


plt.yticks(np.arange(0, 601, 100))

plt.xticks(np.arange(0, 101, 10))

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3.HYPOTHESIS TESTING

To formally examine the relationship between poverty and mortality, we conduct a statistical hypothesis test using the Pearson correlation coefficient.

### Hypotheses

- **H0 (Null Hypothesis):** There is no relationship between poverty rate and mortality rate.
- **H1 (Alternative Hypothesis):** Higher poverty rates are associated with higher mortality rates.

### Method

We use the Pearson correlation coefficient to measure the strength and direction of the relationship, and evaluate statistical significance using the p-value.

### Significance Level

We use a significance level of **alpha = 0.05**.

In [ ]:
from scipy.stats import pearsonr

corr, p_value = pearsonr(data["poverty_rate"], data["mortality_rate"])

print("Correlation:", round(corr, 3))
print("P-value:", p_value)

## Results of Hypothesis Testing

The Pearson correlation coefficient is **0.553**, indicating a moderate positive relationship between poverty and mortality rates.

The p-value is below the significance level of 0.05, meaning the relationship is statistically significant.

### Conclusion

- **Result:** We reject the null hypothesis (H0).
- **Interpretation:** There is strong statistical evidence that counties with higher poverty rates tend to have higher mortality rates.

This suggests that socioeconomic conditions play an important role in health outcomes.

In [ ]:
if p_value < 0.05:
    print("Result: Reject H0 -> Significant relationship")
else:
    print("Result: Fail to reject H0 -> No significant relationship")

## 4. Apply ML Methods and Additional Hypothesis Tests

This section extends the statistical analysis with five hypothesis tests and one machine learning method. The goal is to examine whether poverty and ethnic composition are significantly associated with cardiovascular mortality, and then test whether these variables can predict county-level mortality rates.

### 4.1 Hypotheses

We evaluate five hypotheses using the merged county-level dataset:

1. **H1:** Poverty rate is positively associated with cardiovascular mortality rate.
2. **H2:** Counties with above-median poverty have higher cardiovascular mortality than counties with below-median poverty.
3. **H3:** The percentage of Black population is positively associated with cardiovascular mortality rate.
4. **H4:** The percentage of White population is negatively associated with cardiovascular mortality rate.
5. **H5:** The percentage of Hispanic population is associated with cardiovascular mortality rate.

For correlation-based tests, the null hypothesis states that there is no association between the variables. For the group comparison, the null hypothesis states that the mean mortality rates of high-poverty and low-poverty counties are equal. Statistical significance is evaluated using a significance level of alpha = 0.05.

In [ ]:
from scipy.stats import pearsonr, ttest_ind
import pandas as pd
import numpy as np

analysis_data = data[["poverty_rate", "black", "white", "hispanic", "mortality_rate"]].dropna().copy()

test_results = []
alpha = 0.05

def add_result(hypothesis, test_name, statistic, p_value, effect):
    test_results.append({
        "Hypothesis": hypothesis,
        "Test": test_name,
        "Statistic": statistic,
        "P-value": p_value,
        "Effect / Direction": effect
    })

# H1: Poverty and mortality, Pearson correlation
corr, p_value = pearsonr(analysis_data["poverty_rate"], analysis_data["mortality_rate"])
add_result(
    "H1: Poverty rate vs mortality rate",
    "Pearson correlation",
    corr,
    p_value,
    "Positive correlation"
)

# H2: Mortality difference between high- and low-poverty counties
poverty_median = analysis_data["poverty_rate"].median()
high_poverty = analysis_data.loc[analysis_data["poverty_rate"] >= poverty_median, "mortality_rate"]
low_poverty = analysis_data.loc[analysis_data["poverty_rate"] < poverty_median, "mortality_rate"]
mean_difference = high_poverty.mean() - low_poverty.mean()
t_stat, p_value = ttest_ind(
    high_poverty,
    low_poverty,
    equal_var=False,
    alternative="greater"
)
add_result(
    "H2: High-poverty vs low-poverty mortality",
    "Welch t-test, one-sided",
    t_stat,
    p_value,
    f"High-poverty mean is {mean_difference:.2f} higher"
)

# H3: Black population share and mortality
corr, p_value = pearsonr(analysis_data["black"], analysis_data["mortality_rate"])
add_result(
    "H3: Black population share vs mortality rate",
    "Pearson correlation",
    corr,
    p_value,
    "Positive correlation"
)

# H4: White population share and mortality
corr, p_value = pearsonr(analysis_data["white"], analysis_data["mortality_rate"])
add_result(
    "H4: White population share vs mortality rate",
    "Pearson correlation",
    corr,
    p_value,
    "Negative correlation"
)

# H5: Hispanic population share and mortality
corr, p_value = pearsonr(analysis_data["hispanic"], analysis_data["mortality_rate"])
add_result(
    "H5: Hispanic population share vs mortality rate",
    "Pearson correlation",
    corr,
    p_value,
    "Negative correlation"
)

results_df = pd.DataFrame(test_results)
results_df["Decision"] = np.where(
    results_df["P-value"] < alpha,
    "Reject H0",
    "Fail to reject H0"
)
results_df["Statistic"] = results_df["Statistic"].round(3)
results_df["P-value"] = results_df["P-value"].map(lambda x: f"{x:.3e}")

results_df.style.hide(axis="index").set_caption("Summary of Hypothesis Test Results")

### 4.2 Machine Learning Methods: Linear Regression and Random Forest

To predict cardiovascular mortality rates, we use two regression models. **Linear Regression** serves as an interpretable baseline because its coefficients show the direction and size of each predictor's association with mortality. **Random Forest Regression** is added as a more flexible model that can capture non-linear relationships between poverty, demographic composition, and mortality. The target variable is `mortality_rate`, and the predictors are poverty rate plus the Black, White, and Hispanic population percentages.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

features = ["poverty_rate", "black", "white", "hispanic"]
target = "mortality_rate"

ml_data = data[features + [target]].dropna().copy()
X = ml_data[features]
y = ml_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regression": RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=5,
        random_state=42
    )
}

predictions = {}
model_rows = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[model_name] = y_pred

    model_rows.append({
        "Model": model_name,
        "Mean Absolute Error": mean_absolute_error(y_test, y_pred),
        "Root Mean Squared Error": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R-squared": r2_score(y_test, y_pred)
    })

linear_model = models["Linear Regression"]
rf_model = models["Random Forest Regression"]
best_model_name = max(model_rows, key=lambda row: row["R-squared"])["Model"]
y_pred = predictions[best_model_name]

model_results = pd.DataFrame(model_rows)
model_results[["Mean Absolute Error", "Root Mean Squared Error", "R-squared"]] = model_results[["Mean Absolute Error", "Root Mean Squared Error", "R-squared"]].round(3)
model_results

In [ ]:
coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": linear_model.coef_
}).sort_values("Coefficient", key=lambda s: s.abs(), ascending=False)

rf_importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.barplot(data=coef_df, x="Coefficient", y="Feature", hue="Feature", palette="viridis", legend=False)
plt.axvline(0, color="black", linewidth=1)
plt.title("Linear Regression Coefficients")
plt.xlabel("Coefficient")
plt.ylabel("Predictor")

plt.subplot(1, 2, 2)
sns.barplot(data=rf_importance_df, x="Importance", y="Feature", hue="Feature", palette="mako", legend=False)
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Predictor")

plt.tight_layout()
plt.show()

display(coef_df)
display(rf_importance_df)

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7)
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], color="red", linestyle="--")
plt.title(f"Actual vs Predicted Mortality Rates ({best_model_name})")
plt.xlabel("Actual mortality rate")
plt.ylabel("Predicted mortality rate")
plt.tight_layout()
plt.show()

### 4.3 Interpretation

The hypothesis tests provide formal evidence about the relationships observed in the EDA. If the p-values are below 0.05, the corresponding relationships are considered statistically significant. The machine learning section compares Linear Regression and Random Forest Regression to evaluate whether poverty and ethnic composition together contain predictive information about cardiovascular mortality. The model metrics should be interpreted as follows: lower MAE/RMSE values indicate smaller prediction errors, while a higher R-squared value indicates that the predictors explain more variation in mortality rates.

The Linear Regression coefficient table shows the estimated direction of each predictor while holding the other predictors constant. Positive coefficients indicate that higher values of that predictor are associated with higher predicted mortality, while negative coefficients indicate lower predicted mortality. The Random Forest feature importance table shows which predictors contribute most to the model's predictions, but these importance values do not directly show whether the relationship is positive or negative.